In [1]:
import os, sys
import datetime
import csv
import pandas as pd
import numpy as np
import argparse
from tqdm import tqdm
# from zUtils import zData

import django
#from djCOADD import djOrgDB


In [2]:
djConfig = 'Laptop'

# Django Folder -------------------------------------------------------------
if djConfig == 'Meran':
    djDir = "D:/Code/zdjCode/adjCOADD"
#   uploadDir = "C:/Code/A02_WorkDB/03_Django/adjCOADD/utilities/upload_data/Data"
#   orgdbDir = "C:/Users/uqjzuegg/The University of Queensland/IMB CO-ADD - OrgDB"
elif djConfig == 'Work':
    djDir = "/home/uqjzuegg/xhome/Code/zdjCode/adjCOADD"
#     uploadDir = "C:/Data/A02_WorkDB/03_Django/adjCOADD/utilities/upload_data/Data"
elif djConfig == 'Laptop':
    djDir = "C:/Code/zdjCode/adjCOADD"
#     uploadDir = "/home/uqjzuegg/DeepMicroB/Code/Python/Django/adjCOADD/utilities/upload_data/Data"
else:
    djDir = None

# Django -------------------------------------------------------------
sys.path.append(djDir)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "adjcoadd.settings")

# Needed for Jupyter
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()
from apputil.models import ApplicationUser, Dictionary

print()
print(f"Python         : {sys.version.split('|')[0]}")
print(f"Conda Env      : {os.environ['CONDA_DEFAULT_ENV']}")
#logger.info(f"LogFile        : {logFileName}")

print(f"Django         : {django.__version__}")
print(f"Django Folder  : {djDir}")
print(f"Django Project : {os.environ['DJANGO_SETTINGS_MODULE']}")

Project: adjCOADD 
Version: 1.3.2
Host Name: imb-coadd-db.imb.uq.edu.au

Python         : 3.11.4 
Conda Env      : dj42py311
Django         : 4.2.2
Django Folder  : C:/Code/zdjCode/adjCOADD
Django Project : adjcoadd.settings


In [3]:
from dorganism.models import Organism, Organism_Batch

org_col = ['organism_id','gen_property']
org_qry = Organism.objects.all().values(*org_col)
org_df = pd.DataFrame(list(org_qry), columns=org_col)
print(f"[Organism] : {len(org_df)}")


[Organism] : 1979


In [7]:
from dgene.models import AMR_Genotype
upload = True

for idx, row in tqdm(org_df.iterrows(), total=len(org_df)):
    amrgene_qry = AMR_Genotype.objects.filter(orgbatch_id__organism_id = row['organism_id'], 
                                              orgbatch_id__qc_status = 'Pass',
                                              gene_id__gene_type = 'Resistance', 
                                              ).values_list('amr_method','gene_id__gene_code')
    amrgene_df = pd.DataFrame(list(amrgene_qry), columns=['amr_method','gene_code'])
    if len(amrgene_df)>0:
        amrfinder_lst = amrgene_df[amrgene_df['amr_method'] == 'AMR Finder']['gene_code'].to_list()
        amrcard_lst = amrgene_df[amrgene_df['amr_method'] == 'Abricate card']['gene_code'].to_list()

        genprop_lst = row['gen_property'].split(";")        
        
        if len(amrfinder_lst) > 0:
            amrfinder_str = f"[AMR Finder]: {', '.join(amrfinder_lst)}"
            if '[AMR Finder]' not in row['gen_property']:
                # New entry
                genprop_lst.append(amrfinder_str)
            else:
                # Replace current entry
                for n in range(len(genprop_lst)):
                    if '[AMR Finder]' in genprop_lst[n]:
                        genprop_lst[n] = amrfinder_str

        # if len(amrcard_lst) > 0 :
        #     amrcard_str = f"[Card]: {', '.join(amrcard_lst)}"
        #     if '[Card]' not in row['gen_property']:
        #         genprop_lst.append(amrcard_str)
        #         # New entry
        #         genprop_lst.append(amrcard_str)
        #     else:
        #         # Replace current entry
        #         for n in range(len(genprop_lst)):
        #             if '[Card]' in genprop_lst[n]:
        #                 genprop_lst[n] = amrcard_str

        row['new_gen_property'] = "; ".join(genprop_lst)

        if upload:
            djOrg = Organism.get(row['organism_id'])
            djOrg.gen_property = row['new_gen_property']
            djOrg.save()

        #print(f"{row['organism_id']} AMRGene: {len(amrgene_df)} {len(amrfinder_lst)} {len(amrcard_lst)} Gen_Property {row['gen_property']}")
        #print(f"{row['organism_id']} AMRGene: {row['gen_property']} --> {row['new_gen_property']}")

100%|██████████| 1979/1979 [01:05<00:00, 30.07it/s]
